In [5]:
import rasterio
import numpy as np
from PIL import Image
import os

In [7]:
img_path = r"C:\CS2441\satellite_images\SENTINEL 2\GRANULE\L2A_T44QLJ_A008426_20260417T051527\IMG_DATA\R10m"

In [9]:
patches_path = r"C:\CS2441\satellite_images\patches"

print("water rgb :", os.path.exists(os.path.join(patches_path, "water", "rgb")))
print("water mask :", os.path.exists(os.path.join(patches_path, "water", "mask")))
print("no_water rgb :", os.path.exists(os.path.join(patches_path, "no_water", "rgb")))
print("no_water mask :", os.path.exists(os.path.join(patches_path, "no_water", "mask")))

water rgb : True
water mask : True
no_water rgb : True
no_water mask : True


In [11]:
b02_path = os.path.join(img_path, "T44QLJ_20260417T050651_B02_10m.jp2")
b03_path = os.path.join(img_path, "T44QLJ_20260417T050651_B03_10m.jp2")
b04_path = os.path.join(img_path, "T44QLJ_20260417T050651_B04_10m.jp2")
b08_path = os.path.join(img_path, "T44QLJ_20260417T050651_B08_10m.jp2")

with rasterio.open(b02_path) as src: blue = src.read(1)
with rasterio.open(b03_path) as src: green = src.read(1)
with rasterio.open(b04_path) as src: red = src.read(1)
with rasterio.open(b08_path) as src: nir = src.read(1)

In [12]:
def normalize_percentile(band, low=2, high=98):
    p_low = np.percentile(band, low)
    p_high = np.percentile(band, high)
    clipped = np.clip(band, p_low, p_high)
    return (clipped - p_low) / (p_high - p_low)

true_color = np.dstack((normalize_percentile(red),
                      normalize_percentile(green),
                      normalize_percentile(blue)))

true_color_uint8 = (true_color * 255).astype(np.uint8)

green_f = green.astype(float)
nir_f = nir.astype(float)
ndwi = (green_f - nir_f) / (green_f + nir_f)
water_mask = ndwi > 0

water_mask_uint8 = (water_mask * 255).astype(np.uint8)

print(" True Color Shape : ", true_color_uint8.shape)
print("Water Mask Shape : ", water_mask_uint8.shape)

 True Color Shape :  (10980, 10980, 3)
Water Mask Shape :  (10980, 10980)


In [15]:
patch_size = 128
height, width = water_mask.shape

water_count = 0
no_water_count = 0
patch_num = 0

print(" Starting patch extraction")

for row in range(0, height - patch_size + 1, patch_size):
    for col in range(0, width - patch_size + 1, patch_size):

        rgb_patch  = true_color_uint8[row:row+patch_size, col:col+patch_size, :]
        mask_patch = water_mask_uint8[row:row+patch_size, col:col+patch_size]

        has_water = np.any(mask_patch > 0)

        filename = f"patch_{patch_num:04d}.png"

        if has_water:
            Image.fromarray(rgb_patch).save(
                os.path.join(patches_path, "water", "rgb", filename))
            Image.fromarray(mask_patch).save(
                os.path.join(patches_path, "water", "mask", filename))
            water_count += 1
        else:
            Image.fromarray(rgb_patch).save(
                os.path.join(patches_path, "no_water", "rgb", filename))
            Image.fromarray(mask_patch).save(
                os.path.join(patches_path, "no_water", "mask", filename))
            no_water_count += 1

        patch_num += 1

print("Patch extraction complete!")
print("Total patches    :", patch_num)
print("Water patches    :", water_count)
print("No water patches :", no_water_count)

 Starting patch extraction
Patch extraction complete!
Total patches    : 7225
Water patches    : 3651
No water patches : 3574
